# Local stability from main run

Run the full `GeneralizedPerturbedEquilibrium.main` path for this example, then read the HDF5 output and plot the local `D_I` and ballooning `\Delta'` profiles.

In [ ]:
using Pkg
Pkg.activate("../..")

using GeneralizedPerturbedEquilibrium
using HDF5
using Plots
using LaTeXStrings
using TOML

inputs = TOML.parsefile("gpec.toml")
@assert get(inputs["ForceFreeStates"], "local_stability_flag", false) "local_stability_flag must be true to write local stability output"

GeneralizedPerturbedEquilibrium.main([pwd()])


In [ ]:
h5_name = get(inputs["ForceFreeStates"], "HDF5_filename", "gpec.h5")

psi_norm = h5open(h5_name, "r") do h5
    read(h5["splines/profiles/xs"])
end

di, delta_prime = h5open(h5_name, "r") do h5
    read(h5["locstab/di"]), read(h5["locstab/delta_prime"])
end

println("D_I finite: $(count(isfinite, di)) / $(length(di))")
println("Delta prime finite: $(count(isfinite, delta_prime)) / $(length(delta_prime))")

p_di = plot(
    psi_norm, di;
    xlabel=L"\psi_N",
    ylabel=L"D_I",
    title=L"D_I",
    linewidth=2,
    label="",
    framestyle=:box,
)
hline!(p_di, [0.0]; color=:black, linestyle=:dash, label="")

p_delta = plot(
    psi_norm, delta_prime;
    xlabel=L"\psi_N",
    ylabel=L"\Delta'",
    title=L"\Delta'",
    linewidth=2,
    label="",
    framestyle=:box,
)
hline!(p_delta, [0.0]; color=:black, linestyle=:dash, label="")

display(plot(p_di, p_delta; layout=(1, 2), size=(1050, 390)))
